In [3]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, f1_score, confusion_matrix, ConfusionMatrixDisplay
import json
import matplotlib.pyplot as plt
import os
import joblib

In [4]:
# --- CONFIG ---
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = "../models/models/model_mlp.pt"
SCALER_SAVE_PATH = "../scalers/mlp_scaler.pkl"
RESULTS_DIR = "../outputs/results/mlp"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SCALER_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

In [5]:
# =======================
# 1. Load & Preprocess Data
# =======================
# Giả sử bạn đã lưu preprocessed_data_v4.pkl với các biến:
# X_train, X_val, X_test, y_train, y_val, y_test
X_train, X_val, X_test, y_train, y_val, y_test, *_= joblib.load('../data/processed/preprocessed_data1.pkl')

# Scale dữ liệu
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
joblib.dump(scaler, SCALER_SAVE_PATH)

/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SelectKBest from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


['../scalers/mlp_scaler.pkl']

In [6]:
# =======================
# 2. Dataset & DataLoader
# =======================
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, "values") else y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return {'inputs': self.X[idx], 'labels': self.y[idx]}

BATCH_SIZE = 64
train_ds = TabularDataset(X_train_scaled, y_train)
val_ds   = TabularDataset(X_val_scaled, y_val)
test_ds  = TabularDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)

In [7]:
# =======================
# 3. Simple MLP Model
# =======================
class SimpleMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims=[128, 64, 32], num_classes=2, dropout=0.2):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, num_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

In [8]:
# =======================
# 5. Evaluation
# =======================
def evaluate_mlp(model, test_loader, device, results_dir=RESULTS_DIR):
    os.makedirs(results_dir, exist_ok=True)
    model.eval()
    all_preds, all_probs, all_labels = [], [], []
    with torch.no_grad():
        for batch in test_loader:
            inputs = batch['inputs'].to(device)
            labels = batch['labels'].to(device)
            logits = model(inputs)
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            preds = np.argmax(probs, axis=1)
            all_preds.extend(preds)
            all_probs.extend(probs[:, 1])
            all_labels.extend(labels.cpu().numpy())
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)
    all_labels = np.array(all_labels)
    # Classification report
    report = classification_report(all_labels, all_preds, output_dict=True)
    with open(f'{results_dir}/classification_report.json', 'w') as f:
        json.dump(report, f, indent=4)
    print(classification_report(all_labels, all_preds))
    f1_micro = f1_score(all_labels, all_preds, average='micro')
    print("F1-micro:", f1_micro)
    # Confusion matrix (normalized as percentage)
    cm = confusion_matrix(all_labels, all_preds, normalize='true')
    disp = ConfusionMatrixDisplay(confusion_matrix=cm)
    disp.plot(values_format='.2%')
    plt.title('Confusion Matrix - MLP')
    plt.savefig(f'{results_dir}/confusion_matrix.pdf')
    plt.savefig(f'{results_dir}/confusion_matrix.svg')
    plt.close()

In [9]:
# =======================
# 6. RUN TRAINING & EVAL
# =======================
input_dim = X_train_scaled.shape[1]
mlp_model = SimpleMLP(input_dim, num_classes=2)
# train_mlp(mlp_model, train_loader, val_loader, DEVICE, epochs=30, lr=1e-3, patience=8)

mlp_model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
# <<< Add this line to move the model to the GPU
mlp_model.to(DEVICE)
evaluate_mlp(mlp_model, test_loader, DEVICE, results_dir=RESULTS_DIR)

              precision    recall  f1-score   support

           0       0.99      1.00      0.99     24669
           1       0.93      0.73      0.82      1013

    accuracy                           0.99     25682
   macro avg       0.96      0.87      0.91     25682
weighted avg       0.99      0.99      0.99     25682

F1-micro: 0.9872673467798458


In [10]:
# =========================================
#        1. IMPORTS & CONFIG
# =========================================
import os
import numpy as np
import joblib
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import BertModel
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import json

In [11]:
# --- CONFIG ---
SEQ_LEN = 1         # sequence length (bạn chỉnh theo ý muốn)
INPUT_DIM = 30     # số lượng feature
NUM_CLASSES = 2      # binary classification (attack/benign)
BATCH_SIZE = 64
HIDDEN_SIZE = 768    # secbert-base
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = "../models/models/model_secbert.pt"
SCALER_SAVE_PATH = "../scalers/secbert_scaler.pkl"
RESULTS_DIR = "../outputs/results/secbert"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SCALER_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

In [12]:
# =========================================
#        2. LOAD & PREPROCESS DATA
# =========================================
# Dữ liệu đã tiền xử lý: X_train, X_val, X_test, y_train, y_val, y_test
X_train, X_val, X_test, y_train, y_val, y_test, *_= joblib.load('../data/processed/preprocessed_data1.pkl')

# --- Scale ---
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)
joblib.dump(scaler, SCALER_SAVE_PATH)

# --- Sliding window tạo sequence ---
def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(seq_len, len(X)):
        Xs.append(X[i-seq_len:i])
        ys.append(y[i])
    return np.array(Xs), np.array(ys)

X_train_seq, y_train_seq = create_sequences(X_train_scaled, y_train.values, SEQ_LEN)
X_val_seq, y_val_seq     = create_sequences(X_val_scaled, y_val.values, SEQ_LEN)
X_test_seq, y_test_seq   = create_sequences(X_test_scaled, y_test.values, SEQ_LEN)
print("X_train_scaled shape:", X_train_scaled.shape)

X_train_scaled shape: (119847, 50)


/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SelectKBest from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [13]:
# =========================================
#        3. Dataset & DataLoader
# =========================================
class TimeseriesClassificationDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return {'inputs': self.X[idx], 'labels': self.y[idx]}

train_ds = TimeseriesClassificationDataset(X_train_seq, y_train_seq)
val_ds   = TimeseriesClassificationDataset(X_val_seq, y_val_seq)
test_ds  = TimeseriesClassificationDataset(X_test_seq, y_test_seq)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


In [14]:
# =========================================
#        4. SEC-BERT MODEL
# =========================================
class SecBERTClassifier(nn.Module):
    def __init__(self, input_dim, hidden_size, seq_len, num_classes):
        super().__init__()
        self.embedding = nn.Linear(input_dim, hidden_size)
        pe = self.sinusoidal_positional_embedding(seq_len + 1, hidden_size)
        self.register_buffer("pos_embedding", pe)
        self.cls_token = nn.Parameter(torch.empty(1, 1, hidden_size))
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        self.bert = BertModel.from_pretrained("jackaduma/SecBERT")  # hoặc bert-base-uncased nếu không tải được SecBERT
        self.output_layer = nn.Linear(hidden_size, num_classes)

    def sinusoidal_positional_embedding(self, seq_len, hidden_dim):
        position = torch.arange(0, seq_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, hidden_dim, 2) * (-np.log(10000.0) / hidden_dim))
        pe = torch.zeros(seq_len, hidden_dim)
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0)

    def forward(self, x):
        batch_size = x.size(0)
        x = self.embedding(x)
        cls_expanded = self.cls_token.expand(batch_size, 1, -1)
        x = torch.cat([cls_expanded, x], dim=1)
        x = x + self.pos_embedding[:, :x.size(1)]
        attention_mask = torch.ones(x.size(0), x.size(1), dtype=torch.long, device=x.device)
        outputs = self.bert(inputs_embeds=x, attention_mask=attention_mask)
        cls_state = outputs.last_hidden_state[:, 0, :]
        logits = self.output_layer(cls_state)
        return logits

    def compute_loss(self, predictions, targets):
        return nn.CrossEntropyLoss()(predictions, targets)


In [15]:
model = SecBERTClassifier(input_dim=X_train_scaled.shape[1], hidden_size=768, seq_len=SEQ_LEN, num_classes=2)
model.to(DEVICE)

SecBERTClassifier(
  (embedding): Linear(in_features=50, out_features=768, bias=True)
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(52000, 768, padding_idx=0)
      (position_embeddings): Embedding(514, 768)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-5): 6 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True

In [16]:
# =========================================
#        7. EVALUATE & SAVE METRICS
# =========================================
# Load best model
model.load_state_dict(torch.load(MODEL_SAVE_PATH))
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['inputs'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_preds.extend(preds)
        all_probs.extend(probs[:, 1])  # lấy xác suất class 1 (nếu binary)
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ---- Classification report ----
report = classification_report(all_labels, all_preds, output_dict=True)
with open(f'{RESULTS_DIR}/classification_report.json', 'w') as f:
    json.dump(report, f, indent=4)
print(classification_report(all_labels, all_preds))

# ---- Confusion matrix ----
cm = confusion_matrix(all_labels, all_preds, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format='.2%')
plt.title('Confusion Matrix - SecBERT')
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.pdf')
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.svg')
plt.close()

# ---- ROC Curve ----
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - SecBERT')
plt.legend()
plt.savefig(f'{RESULTS_DIR}/roc_curve.pdf')
plt.savefig(f'{RESULTS_DIR}/roc_curve.svg')
plt.close()

# ---- Precision-Recall Curve ----
precision, recall, _ = precision_recall_curve(all_labels, all_probs)
plt.figure()
plt.plot(recall, precision, label='SecBERT')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - SecBERT')
plt.legend()
plt.savefig(f'{RESULTS_DIR}/pr_curve.pdf')
plt.savefig(f'{RESULTS_DIR}/pr_curve.svg')
plt.close()

print(f"✅ SecBERT pipeline complete! Report & plots saved to: {RESULTS_DIR}")



/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.

              precision    recall  f1-score   support

           0       0.96      1.00      0.98     24668
           1       0.00      0.00      0.00      1013

    accuracy                           0.96     25681
   macro avg       0.48      0.50      0.49     25681
weighted avg       0.92      0.96      0.94     25681

✅ SecBERT pipeline complete! Report & plots saved to: ../outputs/results/secbert


In [17]:
# =========================================
#        1. IMPORTS & CONFIG
# =========================================
import os
import numpy as np
import joblib
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve, ConfusionMatrixDisplay, f1_score
import matplotlib.pyplot as plt
import json
import random

In [18]:
# ----- Fix seed for reproducibility -----
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
set_seed(42)

In [19]:
# --- CONFIG ---
INPUT_DIM = 50        # Sửa đúng số lượng feature bạn còn sau feature selection
NUM_CLASSES = 2
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_SAVE_PATH = "../models/models/model_tabtransformer.pt"
SCALER_SAVE_PATH = "../scalers/tabtransformer_scaler.pkl"
RESULTS_DIR = "../outputs/results/tabtransformer"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(os.path.dirname(SCALER_SAVE_PATH), exist_ok=True)
os.makedirs(os.path.dirname(MODEL_SAVE_PATH), exist_ok=True)

In [20]:
# =========================================
#        2. LOAD & PREPROCESS DATA
# =========================================
# Bạn đã lưu đúng pipeline: nhớ load như sau:
X_train, X_val, X_test, y_train, y_val, y_test, *_= joblib.load('../data/processed/preprocessed_data_v4.pkl')

# --- Scale ---
scaler = MinMaxScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)
X_test_scaled  = scaler.transform(X_test)
joblib.dump(scaler, SCALER_SAVE_PATH)

print("X_train_scaled shape:", X_train_scaled.shape)


X_train_scaled shape: (129664, 50)


/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator VarianceThreshold from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
/home/ics-security/anaconda3/envs/ws1/lib/python3.10/site-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator SelectKBest from version 1.7.1 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [21]:
# =========================================
#        3. Dataset & DataLoader
# =========================================
class TabularDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y.values if hasattr(y, "values") else y, dtype=torch.long)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return {'inputs': self.X[idx], 'labels': self.y[idx]}

train_ds = TabularDataset(X_train_scaled, y_train)
val_ds   = TabularDataset(X_val_scaled, y_val)
test_ds  = TabularDataset(X_test_scaled, y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=False)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, drop_last=False)


In [22]:
# =========================================
#        4. TABULAR TRANSFORMER MODEL
# =========================================
class FeatureTransformerV2(nn.Module):
    def __init__(self,
                 n_feat: int,
                 emb_dim: int = 32,
                 n_layers: int = 6,
                 n_heads: int = 4,
                 ff_factor: int = 4,
                 num_classes: int = 2,
                 dropout: float = 0.1,
                 label_smooth: float = 0.05):
        super().__init__()

        d_model = emb_dim * 2                        # ghép value-emb + col-emb
        self.label_smooth = label_smooth

        # 1) Value embedding (MLP 1→emb_dim)
        self.val_embed = nn.Sequential(
            nn.Linear(1, emb_dim),
            nn.GELU(),
            nn.Linear(emb_dim, emb_dim),
            nn.LayerNorm(emb_dim)
        )
        # 2) Column embedding
        self.col_embed = nn.Embedding(n_feat, emb_dim)

        # 3) CLS token
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))

        # 4) Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=n_heads,
            dim_feedforward=d_model * ff_factor,
            dropout=dropout,
            activation='gelu',
            batch_first=True
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.drop = nn.Dropout(dropout)

        # 5) MLP classification head
        self.head = nn.Sequential(
            nn.Linear(d_model, 256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):                 # x: (B, N_feat)
        B, N = x.size()
        v_emb = self.val_embed(x.unsqueeze(-1))          # (B, N, emb_dim)
        c_emb = self.col_embed.weight[:N]                # (N, emb_dim)
        tokens = torch.cat([v_emb, c_emb.expand(B, -1, -1)], dim=-1)  # (B, N, 2*emb_dim)
        tokens = self.drop(tokens)
        cls = self.cls_token.expand(B, -1, -1)           # (B,1,d_model)
        tok_seq = torch.cat([cls, tokens], dim=1)        # (B, N+1, d_model)
        encoded = self.encoder(tok_seq)                  # (B, N+1, d_model)
        logits  = self.head(encoded[:, 0])               # dùng CLS
        return logits

    def compute_loss(self, logits, targets):
        if self.label_smooth > 0:
            n_class = logits.size(1)
            smoothed_labels = F.one_hot(targets, n_class).float()
            smoothed_labels = smoothed_labels * (1 - self.label_smooth) \
                               + self.label_smooth / n_class
            log_prob = F.log_softmax(logits, dim=1)
            loss = (-smoothed_labels * log_prob).sum(dim=1).mean()
            return loss
        else:
            return nn.CrossEntropyLoss()(logits, targets)


In [23]:
model = FeatureTransformerV2(
    n_feat=X_train_scaled.shape[1],
    emb_dim=32,
    n_layers=6,
    n_heads=4,
    num_classes=2
)
model.to(DEVICE)

FeatureTransformerV2(
  (val_embed): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): LayerNorm((32,), eps=1e-05, elementwise_affine=True)
  )
  (col_embed): Embedding(50, 32)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-5): 6 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=256, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=256, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )


In [24]:
# =========================================
#        7. EVALUATE & SAVE METRICS
# =========================================
model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=DEVICE))
model.eval()

all_preds = []
all_probs = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['inputs'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        logits = model(inputs)
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = np.argmax(probs, axis=1)
        all_preds.extend(preds)
        all_probs.extend(probs[:, 1])  # lấy xác suất class 1 (nếu binary)
        all_labels.extend(labels.cpu().numpy())

all_preds = np.array(all_preds)
all_probs = np.array(all_probs)
all_labels = np.array(all_labels)

# ---- Classification report ----
report = classification_report(all_labels, all_preds, output_dict=True)
with open(f'{RESULTS_DIR}/classification_report.json', 'w') as f:
    json.dump(report, f, indent=4)
print(classification_report(all_labels, all_preds))
f1_micro = f1_score(all_labels, all_preds, average='micro')
print("F1-micro:", f1_micro)

# ---- Confusion matrix ----
cm = confusion_matrix(all_labels, all_preds, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(values_format='.2%')
plt.title('Confusion Matrix - Tabular Transformer')
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.pdf')
plt.savefig(f'{RESULTS_DIR}/confusion_matrix.svg')
plt.close()

# ---- ROC Curve ----
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc_score = roc_auc_score(all_labels, all_probs)
plt.figure()
plt.plot(fpr, tpr, label=f'AUC = {auc_score:.2f}')
plt.plot([0, 1], [0, 1], linestyle='--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Tabular Transformer')
plt.legend()
plt.savefig(f'{RESULTS_DIR}/roc_curve.pdf')
plt.savefig(f'{RESULTS_DIR}/roc_curve.svg')
plt.close()

# ---- Precision-Recall Curve ----
precision, recall, _ = precision_recall_curve(all_labels, all_probs)
plt.figure()
plt.plot(recall, precision, label='Tabular Transformer')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve - Tabular Transformer')
plt.legend()
plt.savefig(f'{RESULTS_DIR}/pr_curve.pdf')
plt.savefig(f'{RESULTS_DIR}/pr_curve.svg')
plt.close()

print(f"✅ Tabular Transformer pipeline complete! Report & plots saved to: {RESULTS_DIR}")


              precision    recall  f1-score   support

           0       0.97      1.00      0.98     24669
           1       0.96      0.77      0.85      3117

    accuracy                           0.97     27786
   macro avg       0.96      0.88      0.92     27786
weighted avg       0.97      0.97      0.97     27786

F1-micro: 0.9701288418628086
✅ Tabular Transformer pipeline complete! Report & plots saved to: ../outputs/results/tabtransformer
